In [39]:
import pandas as pd

In [40]:
#data collection
data = pd.read_csv(r"C:\Users\chaka\Preethu\My_Git_Repo\Final Project_Rideflow\Datasets\preprocessed_rideflow_datasets.csv")
data.columns

Index(['ride_id', 'driver_id', 'customer_id', 'fare_price', 'surge_multiplier',
       'driver_rating', 'customer_rating', 'estimated_eta_min',
       'actual_eta_min', 'ride_status', 'traffic_level', 'driver_active',
       'feedback_text', 'hour', 'day_of_week', 'is_weekend',
       'available_drivers', 'distance_km', 'cancellation_risk', 'is_peak_hour',
       'eta_diff', 'ride_count', 'pickup_zone_Adyar', 'pickup_zone_Anna Nagar',
       'pickup_zone_OMR', 'pickup_zone_Porur', 'pickup_zone_T Nagar',
       'pickup_zone_Tambaram', 'pickup_zone_Velachery', 'drop_zone_Adyar',
       'drop_zone_Anna Nagar', 'drop_zone_OMR', 'drop_zone_Porur',
       'drop_zone_T Nagar', 'drop_zone_Tambaram', 'drop_zone_Velachery',
       'weather_clear', 'weather_cloudy', 'weather_rain'],
      dtype='object')

In [41]:
#inputs
driver_data = pd.DataFrame(data[['driver_id', 'driver_rating', 'cancellation_risk','estimated_eta_min']])
demand = "High"
#convert to dict
drivers = driver_data.to_dict(orient='records')

In [42]:
#scoring function
def compute_score(driver, demand):
    eta = driver["estimated_eta_min"]
    rating = driver["driver_rating"]
    cancel = driver["cancellation_risk"]

    # Normalize factors
    eta_score = 1 / (eta + 1)
    rating_score = rating / 5
    cancel_score = 1 - cancel

    # Demand-based weighting
    if demand == "High":
        score = (0.5 * eta_score) + (0.3 * rating_score) + (0.2 * cancel_score)
    elif demand == "Medium":
        score = (0.3 * eta_score) + (0.4 * rating_score) + (0.3 * cancel_score)
    else:
        score = (0.2 * eta_score) + (0.5 * rating_score) + (0.3 * cancel_score)

    return score

In [43]:
#select best driver
def recommend_driver(drivers, demand):
    for d in drivers:
        d["score"] = compute_score(d, demand)

    best = sorted(drivers, key=lambda x: x["score"], reverse=True)[0]
    return best

In [44]:
#LLM-STYLE EXPLANATION
def explain_recommendation(driver, demand):
    return f"""
🚗 Recommended Driver: {driver['driver_id']}

Reason:
- ETA is {driver['estimated_eta_min']} mins (optimized for {demand} demand)
- High rating of {driver['driver_rating']}
- Low cancellation risk ({driver['cancellation_risk']})

✅ This driver provides the best balance of speed, reliability, and service quality.
"""

In [45]:
best_driver = recommend_driver(drivers, demand)
print(explain_recommendation(best_driver, demand))


🚗 Recommended Driver: 1953

Reason:
- ETA is 2.589181828645964 mins (optimized for High demand)
- High rating of 4.970720155089126
- Low cancellation risk (1.171235993688315)

✅ This driver provides the best balance of speed, reliability, and service quality.

